# 0.1.1 - Statistics

## Setup

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

ROOT = '../../'
RANDOM_SEED = 42  # same as 0.1 and 0.5
RADON_REFERENCE_LEVEL = 300  # Bq/m3

# Same filters as 0.1
MIN_DURATION_DAYS = 90
MIN_CONSTRUCTION_YEAR = 900
MAX_CONSTRUCTION_YEAR = 2025

## Data

In [ ]:
df_raw = pd.read_excel(f'{ROOT}data/raw/01_OFSP_radon_bdd/RDN-BRIEF-1-ADMIN-CACHED-FR.2025-03-05.xlsx')

# Geolocated records, without the duplicates introduced by the spatial joins (index = row of dataset-v1.0)
df_geo = pd.read_csv(f'{ROOT}data/final/dataset-v1.0.csv', low_memory=False).drop_duplicates(subset='orig_idx')

# Modelling dataset written by 0.1 (same index)
X_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-x-train.csv', index_col=0)
X_test = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-x-test.csv', index_col=0)
X = pd.concat([X_train, X_test])
y_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-y-train.csv', index_col=0).squeeze()
y_test = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-y-test.csv', index_col=0).squeeze()
egid = df_geo['EGID']

## Initial database

In [ ]:
# Same date parsing as notebooks/01_data-processing/0.2-process-radon.ipynb
raw_start = pd.to_datetime(df_raw['Début'], format='%d.%m.%Y', errors='coerce')
raw_end = pd.to_datetime(df_raw['Fin'], format='%d.%m.%Y', errors='coerce')

print(f"Period:                     {int(raw_start.dt.year.min())} - {int(raw_end.dt.year.max())}")
print(f"Observations:               {len(df_raw):,}")
print(f"Unique buildings (EGID):    {df_raw['EGID'].nunique():,}")
print(f"Observations without EGID:  {df_raw['EGID'].isna().sum():,}")

## Exclusion funnel

No EGID; not geolocated in RegBL (dropped in `01_data-processing/0.4`); then the filters of 0.1: missing or short (< 90 days) measurements, missing radon concentration, missing or implausible construction year.

In [ ]:
df_pub = df_geo

funnel = []

def log_step(step, data, excluded):
    funnel.append({
        'Step': step,
        'Excluded': excluded,
        'Remaining observations': len(data),
        'Remaining unique EGID': data['EGID'].nunique(),
    })

log_step('Initial database', df_raw, None)

no_egid = df_raw['EGID'].isna()
log_step('Excluded: no EGID', df_raw[~no_egid], no_egid.sum())

not_geolocated = df_raw['EGID'].notna() & ~df_raw['EGID'].isin(set(df_pub['EGID']))
assert len(df_raw) - no_egid.sum() - not_geolocated.sum() == len(df_pub), 'Raw and geolocated records do not reconcile'
log_step('Excluded: not geolocated in RegBL', df_pub, not_geolocated.sum())

step = df_pub['Rdn - duration_days'].notna()
log_step('Excluded: missing measurement duration', df_pub := df_pub[step], (~step).sum())

step = df_pub['Rdn - duration_days'] >= MIN_DURATION_DAYS
log_step('Excluded: duration < 90 days', df_pub := df_pub[step], (~step).sum())

step = df_pub['Rdn - Concentration de radon (Bq/m3)_BQ_M3'].notna()
log_step('Excluded: missing radon concentration', df_pub := df_pub[step], (~step).sum())

step = df_pub['Rdn - Année de construction'].notna()
log_step('Excluded: missing construction year', df_pub := df_pub[step], (~step).sum())

step = df_pub['Rdn - Année de construction'].between(MIN_CONSTRUCTION_YEAR, MAX_CONSTRUCTION_YEAR, inclusive='right')
log_step(f'Excluded: implausible construction year (<= {MIN_CONSTRUCTION_YEAR} or > {MAX_CONSTRUCTION_YEAR})', df_pub := df_pub[step], (~step).sum())

assert len(df_pub) == len(X), 'Funnel does not match the modelling dataset written by 0.1'

funnel = pd.DataFrame(funnel)
funnel

## Final dataset

In [ ]:
print(f"Observations:                      {len(X):,}")
print(f"Unique buildings (EGID):           {df_pub['EGID'].nunique():,}")
print(f"Features:                          {X.shape[1]}")
print(f"Measurement period:                {int(df_pub['Rdn - Début_year'].min())} - {int(df_pub['Rdn - Fin_year'].max())}")

## Data partitions

All splits are by building; the validation subset is split from the training subset as in 0.3 and 0.5.

In [ ]:
egid_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-egid-train.csv', index_col=0)['EGID']
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
fit_idx, val_idx = next(splitter.split(X_train, y_train, groups=egid_train.loc[X_train.index]))

partitions = {'Train': y_train.iloc[fit_idx], 'Validation': y_train.iloc[val_idx], 'Test': y_test}
buildings = {name: set(egid.loc[y_part.index]) for name, y_part in partitions.items()}
assert sum(map(len, buildings.values())) == len(set.union(*buildings.values())), 'Buildings shared between partitions'

pd.DataFrame([
    {
        'Partition': name,
        'Observations': len(y_part),
        f'≥ {RADON_REFERENCE_LEVEL} Bq/m3 (%)': round((y_part >= RADON_REFERENCE_LEVEL).mean() * 100, 1),
        'Unique buildings (EGID)': len(buildings[name]),
    }
    for name, y_part in partitions.items()
])

## Supplementary Table S1

Model predictors with source dataset, description, data provider and unit / type. Saved to `reports/table-s1.csv`.

In [ ]:
# source: (dataset, data source)
SOURCES = {
    'FOPH-DB': ('Swiss radon database (extract 2025-03-05)', 'Federal Office of Public Health (FOPH)'),
    'FOPH-MAP': ('Swiss radon map (2018)', 'Federal Office of Public Health (FOPH)'),
    'FSO-REGBL': ('Federal Register of Buildings and Dwellings (RegBL)', 'Federal Statistical Office (FSO)'),
    'FOEN-URA': ('Uranium concentration map (GCBA_Uran)', 'Federal Office for the Environment (FOEN)'),
    'METEO': ('Climate normals 1991–2020', 'MeteoSwiss'),
    'SWISSTOPO-TMUD': ('Thickness model of unconsolidated deposits (TMUD25, 2021)', 'swisstopo'),
    'SWISSTOPO-DHM': ('Digital height model DHM200', 'swisstopo'),
    'SWISSTOPO-GEOL': ('Geological map of Switzerland 1:500 000', 'swisstopo'),
    'MINERGIE': ('Certified buildings database', 'Minergie Association'),
    'FOEN-GWK': ('Groundwater bodies (hydrogeological water masses)', 'Federal Office for the Environment (FOEN)'),
    'FOEN-SEISM': ('Seismic zones (SIA 261)', 'Federal Office for the Environment (FOEN)'),
    'FOEN-KARST': ('Hydrogeological karst map', 'Federal Office for the Environment (FOEN)'),
    'FOEN-FOREST': ('Forest site regions', 'Federal Office for the Environment (FOEN)'),
}

# feature: (description, unit, source)
FEATURES = {
    'Rdn - Commune': ('Municipality', '–', 'FOPH-DB'),
    'Rdn - Catégorie de bâtiment': ('Building category', '–', 'FOPH-DB'),
    'Rdn - Année de construction': ('Construction year', 'year', 'FOPH-DB'),
    'Rdn - Nombre étages': ('Number of floors', 'count', 'FOPH-DB'),
    'Rdn - Fondation': ('Foundation material', '–', 'FOPH-DB'),
    'Rdn - Structure de fondation': ('Foundation structure', '–', 'FOPH-DB'),
    'Rdn - Nombre étage sous-terrain': ('Number of underground floors', 'count', 'FOPH-DB'),
    'Rdn - Terrain en pente': ('Building on sloping terrain', 'yes/no', 'FOPH-DB'),
    'Rdn - Ventilation contrôlée': ('Mechanical ventilation', 'yes/no', 'FOPH-DB'),
    'Rdn - Type de local': ('Type of room measured', '–', 'FOPH-DB'),
    'Rdn - Etage': ('Floor of the measured room (0 = ground floor)', 'floor', 'FOPH-DB'),
    'Coordonnée E': ('Easting (LV95)', 'm', 'FSO-REGBL'),
    'Coordonnée N': ('Northing (LV95)', 'm', 'FSO-REGBL'),
    'RegBL - Classe de bâtiment': ('Building class', '–', 'FSO-REGBL'),
    'RegBL - Période de construction': ('Construction period', '–', 'FSO-REGBL'),
    'RegBL - Surface du bâtiment': ('Building footprint area', 'm²', 'FSO-REGBL'),
    'RegBL - Volume du bâtiment': ('Building volume', 'm³', 'FSO-REGBL'),
    'RegBL - Nombre de logements': ('Number of dwellings', 'count', 'FSO-REGBL'),
    'RegBL - Nombre de pièces d’hab. indép.': ('Number of separate habitable rooms', 'count', 'FSO-REGBL'),
    'RegBL - Abri de protection civile': ('Civil protection shelter', 'yes/no', 'FSO-REGBL'),
    'RegBL - Surface de référence énergétique': ('Energy reference area', 'm²', 'FSO-REGBL'),
    'RegBL - Générateur de chaleur pour le chauffage 1': ('Heat generator for space heating (primary)', '–', 'FSO-REGBL'),
    'RegBL - Source d’énergie / de chaleur pour le chauffage 1': ('Energy source for space heating (primary)', '–', 'FSO-REGBL'),
    'RegBL - Générateur de chaleur pour le chauffage 2': ('Heat generator for space heating (secondary)', '–', 'FSO-REGBL'),
    'RegBL - Source d’énergie / de chaleur pour le chauffage 2': ('Energy source for space heating (secondary)', '–', 'FSO-REGBL'),
    "RegBL - Générateur de chaleur pour l'eau chaude 1": ('Heat generator for hot water (primary)', '–', 'FSO-REGBL'),
    "RegBL - Source d’énergie / de chaleur pour l'eau chaude 1": ('Energy source for hot water (primary)', '–', 'FSO-REGBL'),
    "RegBL - Générateur de chaleur pour l'eau chaude 2": ('Heat generator for hot water (secondary)', '–', 'FSO-REGBL'),
    "RegBL - Source d’énergie / de chaleur pour l'eau chaude 2": ('Energy source for hot water (secondary)', '–', 'FSO-REGBL'),
    'Ura - Concentration uranium': ('Uranium concentration class', 'class', 'FOEN-URA'),
    'Clim - Durée ensoleillement': ('Annual sunshine duration class', 'class', 'METEO'),
    'Clim - Précipitation': ('Annual precipitation class', 'class', 'METEO'),
    'Clim - Température': ('Mean annual temperature class', 'class', 'METEO'),
    'UncSoil - Classe de sol': ('To be completed', 'To be completed', 'SWISSTOPO-TMUD'),
    'Relief - Altitude (m)': ('Altitude', 'm a.s.l.', 'SWISSTOPO-DHM'),
    'Miner - Standard de minergie': ('Minergie standard', '–', 'MINERGIE'),
    'Rdn Map - Probabilité dépassement valeur réf radon': ('Probability of exceeding 300 Bq/m³', 'probability', 'FOPH-MAP'),
    'Rdn Map - Index de confiance dépassement valeur réf': ('Confidence index of the exceedance probability, expressed as a distance (to be checked)', 'm (to be checked)', 'FOPH-MAP'),
    'HydroWM -  GWKName': ('Groundwater body name', '–', 'FOEN-GWK'),
    'HydroWM -  Flussgebiet': ('River basin', '–', 'FOEN-GWK'),
    'HydroWM -  Grundwasserleitertyp': ('Aquifer type (karst, fissured, porous)', '–', 'FOEN-GWK'),
    'HydroWM -  Naturraum': ('Natural region', '–', 'FOEN-GWK'),
    'HydroWM -  Grundwasserregime': ('Groundwater regime', '–', 'FOEN-GWK'),
    'HydroWM -  Area': ('Groundwater body area', 'km²', 'FOEN-GWK'),
    'HydroWM -  SHAPE_Length': ('Groundwater body perimeter', 'm', 'FOEN-GWK'),
    'Seism - ZONE': ('Seismic zone', '–', 'FOEN-SEISM'),
    'Geol - LITHO': ('Lithology', '–', 'SWISSTOPO-GEOL'),
    'Geol - PERIODE': ('Geological period', '–', 'SWISSTOPO-GEOL'),
    'Geol - EPOQUE': ('Geological epoch', '–', 'SWISSTOPO-GEOL'),
    'Geol - ETAGE': ('Geological stage', '–', 'SWISSTOPO-GEOL'),
    'Geol - LEG_TEC_1': ('Tectonic unit (level 1)', '–', 'SWISSTOPO-GEOL'),
    'Geol - LEG_TEC_2': ('Tectonic unit (level 2)', '–', 'SWISSTOPO-GEOL'),
    'Geol - LEG_TEC_3': ('Tectonic unit (level 3)', '–', 'SWISSTOPO-GEOL'),
    'Geol - AQUIFERE': ('Aquifer type', '–', 'SWISSTOPO-GEOL'),
    'Geol - HYDROGEOL': ('Hydrogeological classification', '–', 'SWISSTOPO-GEOL'),
    'Geol - LITH_PET': ('Petrographic lithology', '–', 'SWISSTOPO-GEOL'),
    'Geol - GENESE': ('Rock genesis', '–', 'SWISSTOPO-GEOL'),
    'Geol - TYPE_ROCHE': ('Rock type', '–', 'SWISSTOPO-GEOL'),
    'Geol - COUVERTURE': ('Quaternary cover and its permeability', '–', 'SWISSTOPO-GEOL'),
    'Geol - Distance to fault (m)': ('Distance to nearest certain tectonic fault (derived)', 'm', 'SWISSTOPO-GEOL'),
    'Karst - BA_Name': ('Karst catchment name', '–', 'FOEN-KARST'),
    'Karst - BA_Alt_min': ('Karst catchment minimum altitude', 'm a.s.l.', 'FOEN-KARST'),
    'Karst - BA_Alt_moy': ('Karst catchment mean altitude', 'm a.s.l.', 'FOEN-KARST'),
    'Karst - BA_Alt_max': ('Karst catchment maximum altitude', 'm a.s.l.', 'FOEN-KARST'),
    'Karst - BA_UStruct': ('To be completed', 'To be completed', 'FOEN-KARST'),
    'Karst - BA_Aqui': ('Karst catchment aquifer', '–', 'FOEN-KARST'),
    'Karst - SHAPE_Length': ('Karst catchment perimeter', 'm', 'FOEN-KARST'),
    'Karst - SHAPE_Area': ('Karst catchment area (polygon)', 'm²', 'FOEN-KARST'),
    'Karst - BA_Surf': ('Karst catchment surface', 'km²', 'FOEN-KARST'),
    'Karst - UB_Type': ('To be completed', 'To be completed', 'FOEN-KARST'),
    'Karst - UB_Flux': ('To be completed', 'To be completed', 'FOEN-KARST'),
    'Karst - UB_Alt_min': ('To be completed', 'm a.s.l.', 'FOEN-KARST'),
    'Karst - UB_Alt_moy': ('To be completed', 'm a.s.l.', 'FOEN-KARST'),
    'Karst - UB_Alt_max': ('To be completed', 'm a.s.l.', 'FOEN-KARST'),
    'Shape_Length_left': ('To be completed', 'm', 'FOEN-KARST'),
    'Shape_Area_left': ('To be completed', 'm²', 'FOEN-KARST'),
    'Karst - NK_Type': ('To be completed', 'To be completed', 'FOEN-KARST'),
    'Shape_Length_right': ('To be completed', 'm', 'FOEN-KARST'),
    'Shape_Area_right': ('To be completed', 'm²', 'FOEN-KARST'),
    'Forest - Region_fr': ('Forest site region', '–', 'FOEN-FOREST'),
    'Forest - Shape_Leng': ('Forest site region perimeter', 'm', 'FOEN-FOREST'),
    'Forest - Shape_Area': ('Forest site region area', 'm²', 'FOEN-FOREST'),
}

assert list(FEATURES) == list(X.columns), 'Table S1 metadata does not match the model features'

rows = []
for col, (description, unit, source) in FEATURES.items():
    dataset, data_source = SOURCES[source]
    var_type = 'Categorical' if X[col].dtype == object else 'Numerical'
    rows.append({
        'Feature': col,
        'Dataset': dataset,
        'Description': description,
        'Data source': data_source,
        'Unit / Type': var_type if unit == '–' else f'{unit} / {var_type}',
    })

table_s1 = pd.DataFrame(rows)
Path(f'{ROOT}reports').mkdir(exist_ok=True)
table_s1.to_csv(f'{ROOT}reports/table-s1.csv', index=False)
table_s1

## Case-study municipalities (Figure 4)

All measurements of the initial database in the two municipalities of section 3.4.

In [ ]:
from evaluation import plot_concentration_distributions

CONCENTRATION_COLUMN = 'Concentration de radon (Bq/m3)_BQ_M3'
CASE_STUDIES = ['La Chaux-de-Fonds', 'Zürich']

for commune in CASE_STUDIES:
    rows = df_raw[df_raw['Commune'] == commune]
    values = rows[CONCENTRATION_COLUMN].dropna()
    print(f"{commune}: {len(rows):,} measurements, {rows['EGID'].nunique():,} buildings (EGID), "
          f"{(values >= RADON_REFERENCE_LEVEL).mean():.1%} ≥ {RADON_REFERENCE_LEVEL} Bq/m3")

plot_concentration_distributions({c: df_raw.loc[df_raw['Commune'] == c, CONCENTRATION_COLUMN].dropna() for c in CASE_STUDIES},
                                 'Indoor radon concentrations in the case-study municipalities', RADON_REFERENCE_LEVEL,
                                 path=f'{ROOT}reports/figure-case-studies-distribution')

### Case-study locations (Figure S1)

Natural Earth 1:10m country boundaries and lakes (public domain), downloaded once to `data/external`.

In [ ]:
from evaluation import plot_case_study_locations

plot_case_study_locations(ROOT, {'La Chaux-de-Fonds': (47.102932, 6.830490), 'Zürich': (47.366921, 8.543125)},
                          path=f'{ROOT}reports/figure-case-studies-locations')